# Mission Control AI

Global Solution 2026.1 - Prompt and Artificial Intelligence

Sistema de monitoramento inteligente para uma missão espacial experimental.

## 1. Instalar e iniciar o Ollama no Google Colab

Execute esta célula apenas no Google Colab. Ela instala o Ollama, inicia o servidor e baixa o modelo Llama 3.2 1B.

In [13]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 9 not upgraded.


In [14]:
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time

subprocess.Popen(["ollama", "serve"])
time.sleep(10)

!ollama pull llama3.2:1b
!pip install ollama -q

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [15]:
import ollama

resposta = ollama.chat(
    model="llama3.2:1b",
    messages=[
        {
            "role": "system",
            "content": "Você é um sistema de controle de missão espacial. Analise os dados e indique alertas."
        },
        {
            "role": "user",
            "content": "Temperatura: 95°C | Energia: 18% | Comunicação: instável. Avalie o status."
        }
    ]
)

print(resposta["message"]["content"])

**Alerta de Status**

O sistema de controle de missão espacial está em alerta. Os dados fornecidos não são consistentes e apresentam falhas significativas.

*   **Temperatura**: 95°C é extremamente alto, indicando que a temperatura no local da missão pode ser muito alta ou que há um problema com a estrutura do veículo.
*   **Energia**: 18% indica que a energia disponível para alimentar o sistema de navegação e outros componentes críticos está abaixo do necessário. Isso pode afetar a capacidade da missão em realizar suas tarefas.
*   **Comunicação**: A comunicação é instável, o que significa que há problemas com a transmissão de dados ou a perda de conexão com as estações controladoras.

**Recomendações**

1.  **Avalie a temperatura localizada na missão e ajuste a configuração do veículo se necessário para reduzir a temperatura.
2.  **Verifique a energia disponível para garantir que todos os componentes críticos estejam funcionando corretamente.
3.  **Controle a comunicação e resolva qu

## 2. Código principal da solução

In [16]:
# Mission Control AI - Global Solution 2026.1
# Disciplina: Prompt and Artificial Intelligence
# Projeto: monitoramento inteligente de uma missao espacial experimental

import random
import time
from datetime import datetime

try:
    import ollama
except Exception:
    ollama = None


def gerar_dados_missao(cenario="aleatorio"):
    """Gera dados simulados da missao espacial."""
    if cenario == "critico":
        return {
            "timestamp": datetime.now().strftime("%d/%m/%Y %H:%M:%S"),
            "temperatura": random.randint(88, 105),
            "energia": random.randint(5, 19),
            "comunicacao": random.choice(["instavel", "offline"]),
            "geracao_solar": random.randint(0, 25),
            "oxigenio": random.randint(55, 72),
            "modulo": random.choice(["Habitat", "Energia", "Comunicacao", "Navegacao"]),
        }

    return {
        "timestamp": datetime.now().strftime("%d/%m/%Y %H:%M:%S"),
        "temperatura": random.randint(18, 87),
        "energia": random.randint(20, 100),
        "comunicacao": random.choice(["estavel", "estavel", "instavel"]),
        "geracao_solar": random.randint(10, 95),
        "oxigenio": random.randint(73, 100),
        "modulo": random.choice(["Habitat", "Energia", "Comunicacao", "Navegacao"]),
    }


def gerar_alertas(dados):
    """Aplica regras de decisao para identificar alertas e acoes automaticas."""
    alertas = []
    acoes = []

    if dados["temperatura"] >= 85:
        alertas.append("Temperatura critica no modulo")
        acoes.append("Ativar resfriamento de emergencia")
    elif dados["temperatura"] >= 70:
        alertas.append("Temperatura elevada")
        acoes.append("Aumentar ventilacao e monitorar modulo")

    if dados["energia"] < 20:
        alertas.append("Energia abaixo do limite seguro")
        acoes.append("Ativar modo economia e desligar sistemas nao essenciais")
    elif dados["energia"] < 35:
        alertas.append("Energia em atencao")
        acoes.append("Reduzir consumo e priorizar suporte de vida")

    if dados["comunicacao"] == "offline":
        alertas.append("Comunicacao perdida")
        acoes.append("Reiniciar antena principal e ativar canal reserva")
    elif dados["comunicacao"] == "instavel":
        alertas.append("Sinal de comunicacao instavel")
        acoes.append("Ajustar antena e reenviar pacote de telemetria")

    if dados["oxigenio"] < 70:
        alertas.append("Nivel de oxigenio critico")
        acoes.append("Ativar protocolo de suporte de vida")

    if not alertas:
        alertas.append("Nenhum alerta critico identificado")
        acoes.append("Manter monitoramento padrao da missao")

    return alertas, acoes


def montar_prompt(dados, alertas, acoes):
    return f"""
Analise os dados de telemetria da missao espacial experimental.
Dados:
- Modulo: {dados['modulo']}
- Temperatura: {dados['temperatura']} graus Celsius
- Energia: {dados['energia']}%
- Comunicacao: {dados['comunicacao']}
- Geracao solar: {dados['geracao_solar']}%
- Oxigenio: {dados['oxigenio']}%

Alertas automaticos: {', '.join(alertas)}
Acoes recomendadas pelo sistema: {', '.join(acoes)}

Responda em portugues, de forma objetiva, com:
1. Status geral da missao
2. Risco principal
3. Acao imediata recomendada
""".strip()


def analisar_com_ia(dados, alertas, acoes):
    """Usa Llama via Ollama. Se nao estiver disponivel, usa fallback para manter a demonstracao funcional."""
    prompt_usuario = montar_prompt(dados, alertas, acoes)

    if ollama is not None:
        try:
            resposta = ollama.chat(
                model="llama3.2:1b",
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Voce e uma IA de controle de missao espacial. "
                            "Analise telemetria, identifique riscos e recomende acoes de seguranca."
                        ),
                    },
                    {"role": "user", "content": prompt_usuario},
                ],
            )
            return resposta["message"]["content"]
        except Exception as erro:
            return f"IA indisponivel no momento ({erro}). Analise local: {analise_local(dados, alertas, acoes)}"

    return analise_local(dados, alertas, acoes)


def analise_local(dados, alertas, acoes):
    """Fallback simples para testar o projeto sem Ollama instalado."""
    status = "CRITICO" if any(x in alertas for x in ["Temperatura critica no modulo", "Energia abaixo do limite seguro", "Comunicacao perdida", "Nivel de oxigenio critico"]) else "OPERACIONAL"
    return (
        f"Status geral: {status}. "
        f"Risco principal: {alertas[0]}. "
        f"Acao imediata: {acoes[0]}."
    )


def exibir_status(dados, alertas, acoes, analise_ia):
    print("=" * 60)
    print("MISSION CONTROL AI - STATUS ATUAL")
    print("=" * 60)
    print(f"Horario.........: {dados['timestamp']}")
    print(f"Modulo..........: {dados['modulo']}")
    print(f"Temperatura.....: {dados['temperatura']} C")
    print(f"Energia.........: {dados['energia']}%")
    print(f"Comunicacao.....: {dados['comunicacao']}")
    print(f"Geracao solar...: {dados['geracao_solar']}%")
    print(f"Oxigenio........: {dados['oxigenio']}%")
    print("-" * 60)
    print("ALERTAS AUTOMATICOS")
    for alerta in alertas:
        print(f"- {alerta}")
    print("-" * 60)
    print("ACOES AUTOMATIZADAS")
    for acao in acoes:
        print(f"- {acao}")
    print("-" * 60)
    print("ANALISE DA IA")
    print(analise_ia)
    print("=" * 60)


def executar_monitoramento(cenario="aleatorio", ciclos=3, intervalo=1):
    for ciclo in range(1, ciclos + 1):
        print(f"\nCICLO DE MONITORAMENTO {ciclo}/{ciclos}")
        dados = gerar_dados_missao(cenario if ciclo == ciclos else "aleatorio")
        alertas, acoes = gerar_alertas(dados)
        analise_ia = analisar_com_ia(dados, alertas, acoes)
        exibir_status(dados, alertas, acoes, analise_ia)
        time.sleep(intervalo)


if __name__ == "__main__":
    executar_monitoramento(cenario="critico", ciclos=3, intervalo=1)



CICLO DE MONITORAMENTO 1/3
MISSION CONTROL AI - STATUS ATUAL
Horario.........: 01/06/2026 23:04:14
Modulo..........: Navegacao
Temperatura.....: 38 C
Energia.........: 61%
Comunicacao.....: estavel
Geracao solar...: 52%
Oxigenio........: 97%
------------------------------------------------------------
ALERTAS AUTOMATICOS
- Nenhum alerta critico identificado
------------------------------------------------------------
ACOES AUTOMATIZADAS
- Manter monitoramento padrao da missao
------------------------------------------------------------
ANALISE DA IA
**Status Geral da Missao**

A análise dos dados de telemetria indica que a missao está funcionando corretamente e sem incidentes significativos. Os valores de temperatura, energia e oxigenio estão dentro do limite esperado para a missao.

**Risco Principal**

O risco principal é devido à alta temperatura (38°C), que pode afetar o funcionamento da missao. A temperatura elevada pode causar danos ao equipamento elétrico e aumentar o risco de 

## 3. Teste com cenário normal

In [ ]:
dados = gerar_dados_missao("aleatorio")
alertas, acoes = gerar_alertas(dados)
analise_ia = analisar_com_ia(dados, alertas, acoes)
exibir_status(dados, alertas, acoes, analise_ia)

MISSION CONTROL AI - STATUS ATUAL
Horario.........: 01/06/2026 22:19:03
Modulo..........: Habitat
Temperatura.....: 38 C
Energia.........: 65%
Comunicacao.....: estavel
Geracao solar...: 80%
Oxigenio........: 99%
------------------------------------------------------------
ALERTAS AUTOMATICOS
- Nenhum alerta critico identificado
------------------------------------------------------------
ACOES AUTOMATIZADAS
- Manter monitoramento padrao da missao
------------------------------------------------------------
ANALISE DA IA
IA indisponivel no momento (Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download). Analise local: Status geral: OPERACIONAL. Risco principal: Nenhum alerta critico identificado. Acao imediata: Manter monitoramento padrao da missao.


## 4. Teste com cenário crítico

In [ ]:
dados = gerar_dados_missao("critico")
alertas, acoes = gerar_alertas(dados)
analise_ia = analisar_com_ia(dados, alertas, acoes)
exibir_status(dados, alertas, acoes, analise_ia)

MISSION CONTROL AI - STATUS ATUAL
Horario.........: 01/06/2026 22:19:06
Modulo..........: Energia
Temperatura.....: 104 C
Energia.........: 7%
Comunicacao.....: instavel
Geracao solar...: 9%
Oxigenio........: 65%
------------------------------------------------------------
ALERTAS AUTOMATICOS
- Temperatura critica no modulo
- Energia abaixo do limite seguro
- Sinal de comunicacao instavel
- Nivel de oxigenio critico
------------------------------------------------------------
ACOES AUTOMATIZADAS
- Ativar resfriamento de emergencia
- Ativar modo economia e desligar sistemas nao essenciais
- Ajustar antena e reenviar pacote de telemetria
- Ativar protocolo de suporte de vida
------------------------------------------------------------
ANALISE DA IA
IA indisponivel no momento (Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download). Analise local: Status geral: CRITICO. Risco principal: Temperatura critica no modulo. Acao i

## 5. Monitoramento em ciclos

Executa 3 ciclos, sendo o último com cenário crítico para demonstrar os alertas.

In [17]:
executar_monitoramento(cenario="critico", ciclos=3, intervalo=1)


CICLO DE MONITORAMENTO 1/3
MISSION CONTROL AI - STATUS ATUAL
Horario.........: 01/06/2026 23:30:32
Modulo..........: Comunicacao
Temperatura.....: 77 C
Energia.........: 40%
Comunicacao.....: estavel
Geracao solar...: 90%
Oxigenio........: 93%
------------------------------------------------------------
ALERTAS AUTOMATICOS
- Temperatura elevada
------------------------------------------------------------
ACOES AUTOMATIZADAS
- Aumentar ventilacao e monitorar modulo
------------------------------------------------------------
ANALISE DA IA
**Status Geral da Missão Espacial Experimental**

Agora que analisou os dados da telemetria, percebiu alguns pontos importantes.

*   **Risco Principal**: A temperatura elevada é o risco principal atualmente, pois pode afetar a funcionamento do modulo e causar danos irreparáveis. Além disso, essa condição também pode comprometer a segurança da equipe operadora.
*   **Ações Imediatas Recomendadas**: A seguir, as ações recomendadas pelo sistema são:
   